In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Run this first in a separate cell
!pip install -q torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.8 MB/s eta 0:00:00


In [ ]:
import torch
import random
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────
BASE_MODEL_ID  = "Qwen/Qwen2.5-1.5B"
SFT_PATH    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_PATH   = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/grpo-checkpoint-epoch-1"
#SFT_PATH       = "/content/drive/MyDrive/RL Assignement/sft"    # ← folder with SFT LoRA adapter files
#GRPO_PATH      = "/content/drive/MyDrive/RL Assignement/qwen-math-grpo/checkpoint-93"  # ← folder with GRPO adapter files
MAX_NEW_TOKENS = 512
SEED           = 42

# ── Load a random GSM8K test example ─────────────────────────
random.seed(SEED)
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
example = gsm8k[5]
question = example["question"]
gold_answer = example["answer"].split("####")[-1].strip()

print("=" * 70)
print("📝 PROBLEM")
print("=" * 70)
print(question)
print(f"\n🎯 Gold Answer: {gold_answer}")
print("=" * 70)

# ── Prompt template ───────────────────────────────────────────
def make_prompt(question):
    return (
        "You are a math reasoning assistant. "
        "Solve the problem step by step, then give your final answer as a number.\n\n"
        f"Problem: {question}\n\n"
        "Solution:"
    )

# ── Inference helper ──────────────────────────────────────────
def run_inference(model, tokenizer, question, label):
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = MAX_NEW_TOKENS,
            do_sample         = False,          # greedy for fair comparison
            temperature       = 1.0,
            pad_token_id      = tokenizer.eos_token_id,
        )

    # Decode only the new tokens (strip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print(f"\n{'=' * 70}")
    print(f"🤖 {label}")
    print(f"{'=' * 70}")
    print(response)
    print(f"\n{'─' * 70}")
    return response

# ── Load and run BASELINE (no adapters) ───────────────────────
print("\n⏳ Loading BASE model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype   = torch.bfloat16,
    device_map    = "auto",
    trust_remote_code = True,
)
base_model.config.use_cache = True
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"
print("✅ Base model loaded")

base_response = run_inference(base_model, tokenizer, question, "BASELINE (no fine-tuning)")

# ── Load and run SFT model ─────────────────────────────────────
print("\n⏳ Loading SFT model...")
sft_model = PeftModel.from_pretrained(
    base_model,
    SFT_PATH,
    is_trainable      = False,
    local_files_only  = True,
)
sft_model.eval()
print("✅ SFT model loaded")

sft_response = run_inference(sft_model, tokenizer, question, "SFT MODEL (NuminaMath-CoT fine-tuned)")

# ── Unload SFT, load GRPO ──────────────────────────────────────
print("\n⏳ Loading GRPO model...")
# Detach SFT adapter and load GRPO adapter
sft_model.unload()  # removes LoRA, back to base weights

grpo_model = PeftModel.from_pretrained(
    base_model,
    GRPO_PATH,
    is_trainable     = False,
    local_files_only = True,
)
grpo_model.eval()
print("✅ GRPO model loaded")

grpo_response = run_inference(grpo_model, tokenizer, question, "GRPO MODEL (GSM8k RL fine-tuned)")

# ── Summary ───────────────────────────────────────────────────
import re

def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else "not found"

print(f"\n{'=' * 70}")
print("📊 ANSWER SUMMARY")
print(f"{'=' * 70}")
print(f"  Gold answer  : {gold_answer}")
print(f"  Baseline     : {extract_number(base_response)}")
print(f"  SFT          : {extract_number(sft_response)}")
print(f"  GRPO         : {extract_number(grpo_response)}")
print(f"{'=' * 70}")

📝 PROBLEM
Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?

🎯 Gold Answer: 64

⏳ Loading BASE model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Base model loaded

🤖 BASELINE (no fine-tuning)
 Step 1: Determine the price of the first glass.
The first glass costs $5.

Step 2: Determine the price of the second glass.
The second glass costs 60% of the price of the first glass. To find 60% of $5, we multiply $5 by 0.60:
$5 * 0.60 = $3

Step 3: Determine the price of the third glass.
The third glass also costs 60% of the price of the first glass. To find 60% of $5, we multiply $5 by 0.60:
$5 * 0.60 = $3

Step 4: Determine the price of the fourth glass.
The fourth glass also costs 60% of the price of the first glass. To find 60% of $5, we multiply $5 by 0.60:
$5 * 0.60 = $3

Step 5: Determine the price of the fifth glass.
The fifth glass also costs 60% of the price of the first glass. To find 60% of $5, we multiply $5 by 0.60:
$5 * 0.60 = $3

Step 6: Determine the price of the sixth glass.
The sixth glass also costs 60% of the price of the first glass. To find 60% of $5, we multiply $5 by 0.60:
$5 * 0.60 = $3

Step 7: Determine the